# Screening every Sentinel-2 acquisition over the lake, 2018 to 2025

Checks all Sentinel-2 L2A acquisitions from December 2018 to March 2025 over the 5 km RiskFlow window with the scene classification layer (SCL), to find clear scenes that are not in the archive. A date is clear when at least 95 % of the window has data and at most 10 % is cloud, thin cirrus or cloud shadow; snow does not count as cloud. Only the window itself is read.

**Output** (`Shisper_S2_screening`): `scene_screening.csv` and true-colour thumbnails of every date within 30 days after an outburst. CPU runtime, 5 to 15 minutes.

## Setup

In [ ]:
%pip install -q pystac-client planetary-computer rasterio

import os, re, json, time
from concurrent.futures import ThreadPoolExecutor
import numpy as np, pandas as pd
import pystac_client, planetary_computer, rasterio
from rasterio.enums import Resampling
from rasterio.warp import transform_bounds
from rasterio.windows import from_bounds
import matplotlib.pyplot as plt

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass

DRIVE = '/content/drive/MyDrive'
SCENE_DIR = f'{DRIVE}/Shisper_lake_S2'            # exported archive scenes (notebook 01)
OUT = f'{DRIVE}/Shisper_S2_screening'
os.makedirs(OUT, exist_ok=True)

LAKE = (36.36061, 74.56276)
if os.path.exists(f'{SCENE_DIR}/lake_location.json'):
    j = json.load(open(f'{SCENE_DIR}/lake_location.json')); LAKE = (j['lat'], j['lon'])
KM = 5.0
START, END = '2018-12-01', '2025-03-31'
OUTBURSTS = ['2019-06-22', '2020-05-29', '2021-05-15', '2022-05-07']   # Lovell and Muhammad (2025)
WINDOW_DAYS = 30
MAX_CLOUD, MIN_VALID = 0.10, 0.95
CLOUD = [3, 8, 9, 10]                             # SCL: cloud shadow, cloud medium and high probability, thin cirrus
dlat = KM / 2 / 111.32
dlon = dlat / np.cos(np.radians(LAKE[0]))
BOX = (LAKE[1] - dlon, LAKE[0] - dlat, LAKE[1] + dlon, LAKE[0] + dlat)
pat = re.compile(r'^(\d{4}-\d{2}-\d{2})\.tiff?$', re.I)
ARCHIVE = {m.group(1) for f in (os.listdir(SCENE_DIR) if os.path.isdir(SCENE_DIR) else []) if (m := pat.match(f))}
print('lake', LAKE, '| archive dates:', len(ARCHIVE))
catalog = pystac_client.Client.open('https://planetarycomputer.microsoft.com/api/stac/v1',
                                    modifier=planetary_computer.sign_inplace)

## Cloud cover of every acquisition

In [ ]:
items = []
for y0 in range(int(START[:4]), int(END[:4]) + 1):
    a, b = max(START, f'{y0}-01-01'), min(END, f'{y0}-12-31')
    for k in range(4):
        try:
            items += list(catalog.search(collections=['sentinel-2-l2a'], bbox=BOX, datetime=f'{a}/{b}', limit=1000).items())
            break
        except Exception:
            if k == 3:
                raise
            time.sleep(10 * (k + 1))
print(len(items), 'products cover the window')

NPX = int(round(KM * 1000 / 20))                  # SCL is at 20 m
def read_window(it, key, shape, bands=1):
    href = it.assets[key].href
    for k in range(4):
        try:
            with rasterio.Env(GDAL_DISABLE_READDIR_ON_OPEN='EMPTY_DIR', GDAL_HTTP_MAX_RETRY='5'):
                with rasterio.open(href) as src:
                    w = from_bounds(*transform_bounds('EPSG:4326', src.crs, *BOX), transform=src.transform)
                    idx = 1 if bands == 1 else list(range(1, bands + 1))
                    return src.read(idx, window=w, out_shape=shape, boundless=True, fill_value=0,
                                    resampling=Resampling.nearest)
        except Exception:
            if k == 3:
                return None
            time.sleep(3 * (k + 1))

def day(it):
    return it.datetime.strftime('%Y-%m-%d')

failed, MOS, TILE_CC, NPROD = 0, {}, {}, {}
with ThreadPoolExecutor(16) as pool:
    for it, a in zip(items, pool.map(lambda it: read_window(it, 'SCL', (NPX, NPX)), items)):
        if a is None:
            failed += 1
            continue
        d = day(it)
        MOS[d] = a if d not in MOS else np.where(MOS[d] == 0, a, MOS[d])    # same-day products fill each other's gaps
        TILE_CC.setdefault(d, []).append(it.properties.get('eo:cloud_cover', np.nan))
        NPROD[d] = NPROD.get(d, 0) + 1
print(f'{len(items) - failed} windows read, {failed} failed')

ob = [pd.Timestamp(o) for o in OUTBURSTS]
rows = []
for d, m in sorted(MOS.items()):
    valid = m > 0
    nv = max(int(valid.sum()), 1)
    since = [(pd.Timestamp(d) - o).days for o in ob if pd.Timestamp(d) >= o]
    s = min(since) if since else np.nan
    rows.append(dict(date=d, products=NPROD[d], valid=round(valid.mean(), 3),
                     cloud=round(float(np.isin(m, CLOUD).sum()) / nv, 3), snow=round(float((m == 11).sum()) / nv, 3),
                     scene_cloud_pct=round(float(np.nanmin(TILE_CC[d])), 1), in_archive=d in ARCHIVE,
                     days_since_outburst=s, glof=bool(s <= WINDOW_DAYS)))
T = pd.DataFrame(rows)
T['clear'] = (T.valid >= MIN_VALID) & (T.cloud <= MAX_CLOUD)
T.to_csv(f'{OUT}/scene_screening.csv', index=False)

## Thumbnails of the dates after each outburst

In [ ]:
W = T[T.glof].reset_index(drop=True)
by_day, WD = {}, set(W.date)
for it in items:
    if day(it) in WD and 'visual' in it.assets:
        by_day.setdefault(day(it), []).append(it)

def thumb(d):
    out = None
    for it in by_day.get(d, []):
        a = read_window(it, 'visual', (3, 200, 200), bands=3)
        if a is not None:
            out = a if out is None else np.where(out == 0, a, out)
    return d, out

with ThreadPoolExecutor(8) as pool:
    TH = dict(pool.map(thumb, list(W.date)))
cols = 6
rws = max(1, int(np.ceil(len(W) / cols)))
fig, ax = plt.subplots(rws, cols, figsize=(3 * cols, 3.8 * rws), squeeze=False)
for k in range(rws * cols):
    a = ax[k // cols, k % cols]
    a.set_xticks([]); a.set_yticks([])
    if k >= len(W):
        a.axis('off')
        continue
    r = W.iloc[k]
    if TH.get(r.date) is not None:
        a.imshow(np.transpose(TH[r.date], (1, 2, 0)))
    status = 'in archive' if r.in_archive else ('not in archive, clear' if r.clear else 'not clear')
    a.set_title(f'{r.date} (+{int(r.days_since_outburst)} d), {status}\ncloud {r.cloud:.0%}, data {r.valid:.0%}', fontsize=9)
plt.tight_layout(h_pad=2.5); plt.savefig(f'{OUT}/glof_window_thumbnails.png', dpi=90); plt.show()

## Summary

In [ ]:
print(f'dates with data: {len(T)} | clear: {int(T.clear.sum())} | clear and in the archive: {int((T.clear & T.in_archive).sum())} '
      f'| clear and not in the archive: {int((T.clear & ~T.in_archive).sum())}')
for o in OUTBURSTS:
    t = pd.to_datetime(T.date)
    w = T[(t >= pd.Timestamp(o)) & (t <= pd.Timestamp(o) + pd.Timedelta(days=WINDOW_DAYS))]
    print(f'{o}: {len(w)} dates in the next 30 days | clear {int(w.clear.sum())} | in archive {int(w.in_archive.sum())} '
          f'| clear and not in archive {int((w.clear & ~w.in_archive).sum())}')
print(T[T.glof][['date', 'days_since_outburst', 'valid', 'cloud', 'snow', 'in_archive', 'clear']].to_string(index=False))